In [1]:

#!/usr/bin/env python
# coding: utf-8

# ============================================================
# EGH490 VIRTUAL POWER PLANT DEMONSTRATOR
# ============================================================
#
# Main simulation and hardware-demonstration script.
#
# The simulated VPP consists of five residential premises
# containing different combinations of:
#   - household electrical demand,
#   - rooftop solar PV, and
#   - battery energy storage.
#
# The hospital load and wind/DFIG model are external
# demonstration elements and are not considered VPP assets.
#
# Simulation resolution: 30 minutes.
# The profile data repeats continuously during hardware
# demonstrations.
# ============================================================


# ============================================================
# IMPORTS AND PROJECT CONFIGURATION
# ============================================================

import os
import sys
import time

# Absolute project root on the Jetson.
# Adding this directory to sys.path allows project modules,
# such as the hardware interface, to be imported when the
# notebook/script is launched from another working directory.

PROJECT_ROOT = "/home/jpobar/egh490_vpp_project"

# Make project folders importable
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Startup information used to confirm that the expected
# project directory and Python import path are being used.
print("Working directory:", os.getcwd())
print("Project root added:", PROJECT_ROOT)

import numpy as np
import pandas as pd
from datetime import datetime

# --------------------------------------------------
# IMPORT REQUIRED FUNCTIONS AND SCRIPTS FROM FOLDERS
# --------------------------------------------------

# from scenarios.scenario_setup import (
#     create_coordinated_vpp_scenario
# )

from scenarios.scenario_setup import (
    create_coordinated_vpp_scenario,
    create_independent_dynamic_scenario,
    create_independent_fixed_scenario,
)

from simulation.results_export import (
    export_comparison_results
)

from economics.tariffs import (
    BATTERY_STARTER,
    BATTERY_LITE
)



from hardware.hardware_interface import HardwareInterface
print("HardwareInterface imported successfully")





# ============================================================
# PROFILE DATA
# ============================================================
# Load the prepared seasonal datasets containing the
# 30-minute household demand and PV generation profiles.
#
# Both datasets are loaded here so the simulation can later
# support seasonal scenario selection. The current scenario
# below uses the summer dataset.


summer_data = pd.read_csv(
    "SUMMER Generation and Load profiles.csv"
)

winter_data = pd.read_csv(
    "WINTER Generation and Load profiles.csv"
)

# Basic validation of the selected profile dataset.
# These checks help identify missing data or unexpected CSV
# structure before simulation objects are created.

print("\n=== SUMMER PROFILE DATA CHECK ===")
print(f"Rows loaded: {len(summer_data)}")
print(f"Columns loaded: {len(summer_data.columns)}")
print(f"Missing values: {summer_data.isna().sum().sum()}")

print("\nFirst 5 rows:")
print(summer_data.head())

print("\nColumn names:")
print(summer_data.columns.tolist())


Working directory: /home/jpobar/egh490_vpp_project/Notebooks
Project root added: /home/jpobar/egh490_vpp_project
HardwareInterface imported successfully

=== SUMMER PROFILE DATA CHECK ===
Rows loaded: 336
Columns loaded: 10
Missing values: 0

First 5 rows:
  LOAD_SOURCE_TIMESTEP PV_SOURCE_TIMESTAMP  PV_6.6kW_kW  PV_13.3kW_kW  \
0      04/02/2013 0:00     07/01/2013 0:00          0.0           0.0   
1      04/02/2013 0:30     07/01/2013 0:30          0.0           0.0   
2      04/02/2013 1:00     07/01/2013 1:00          0.0           0.0   
3      04/02/2013 1:30     07/01/2013 1:30          0.0           0.0   
4      04/02/2013 2:00     07/01/2013 2:00          0.0           0.0   

   PV_20kW_kW  H1_HI_B_kW  H2_MED_B_kW  H3_LOW_A_Kw  H4_Hi_A_kW  H5_Med_A_kW  
0         0.0       0.706        0.150        0.135       0.213        0.353  
1         0.0       0.695        0.154        0.136       0.216        0.390  
2         0.0       0.646        0.177        0.059       0.202    

Define simulation Objects


In [2]:
def get_hardware_states(simulation):
    """
    Convert the latest simulated battery results
    into states for the physical LEDs.
    """

    states = {}

    for house in simulation.houses:

        # House 3 has no battery
        if house.battery is None:
            states[house.name] = {
                "state": "no_battery",
                "soc_percent": 0.0
            }
            
            continue

        # Get this house's result from the timestep
        # that has just been simulated
        latest = simulation.house_results_log[house.name][-1]

        soc_kwh = latest["battery_soc_kwh"]
        charge_kw = latest["battery_charge_kw"]
        discharge_kw = latest["battery_discharge_kw"]
        soc_percent = latest["battery_soc_percent"]

        max_soc_kwh = house.battery.capacity_kwh
        min_soc_kwh = house.battery.min_soc_kwh

        # Small tolerance avoids floating-point comparison problems
        tolerance = 1e-6

        if soc_kwh >= max_soc_kwh - tolerance:
            state = "full"

        elif soc_kwh <= min_soc_kwh + tolerance:
            state = "minimum"

        elif charge_kw > tolerance:
            state = "charging"

        elif discharge_kw > tolerance:
            state = "discharging"

        else:
            state = "idle"

        states[house.name] = {
            "state": state,
            "soc_percent": soc_percent
        }

    return states

Simulation Loop

In [3]:
# ============================================================
# REALISTIC PROFILE DATA
# ============================================================

timestep_hours = 0.5
num_steps = len(summer_data)
simulation_hours = num_steps * timestep_hours


# ============================================================
# EXTERNAL DEMONSTRATION PLACEHOLDER PROFILES
# ============================================================

hospital_load_profile = np.full(
    num_steps,
    6.0
)

wind_generation_profile = np.zeros(
    num_steps
)


# ----------------------------
# Household load profiles
# ----------------------------

house_1_load = summer_data["H1_HI_B_kW"].to_numpy()
house_2_load = summer_data["H2_MED_B_kW"].to_numpy()
house_3_load = summer_data["H3_LOW_A_Kw"].to_numpy()
house_4_load = summer_data["H4_Hi_A_kW"].to_numpy()
house_5_load = summer_data["H5_Med_A_kW"].to_numpy()


# ----------------------------
# PV generation profiles
# ----------------------------

house_1_pv = summer_data["PV_13.3kW_kW"].to_numpy()
house_2_pv = summer_data["PV_6.6kW_kW"].to_numpy()
house_3_pv = summer_data["PV_13.3kW_kW"].to_numpy()
house_4_pv = summer_data["PV_20kW_kW"].to_numpy()

house_5_pv = np.zeros(num_steps)

load_timestamps = pd.to_datetime(
    summer_data["LOAD_SOURCE_TIMESTEP"],
    dayfirst=True
)


pv_timestamps = pd.to_datetime(
    summer_data["PV_SOURCE_TIMESTAMP"],
    dayfirst=True
)

print("\n=== TIMESTAMP CHECK ===")
print("Load start:", load_timestamps.iloc[0])
print("Load end:  ", load_timestamps.iloc[-1])
print("PV start:  ", pv_timestamps.iloc[0])
print("PV end:    ", pv_timestamps.iloc[-1])

# Export Limit Profiles

# dynamic_export_profile_kw_per_phase = (
#     summer_data[
#         "Dynamic_Export_Limit_kW_per_phase"
#     ].to_numpy()
# )

daily_dynamic_export_kw_per_phase = np.array(
    [10.0] * 12 +   # 00:00–06:00
    [6.0] * 6  +   # 06:00–09:00
    [1.5] * 12 +   # 09:00–15:00
    [10.0] * 6  +   # 15:00–18:00
    [10.0] * 12     # 18:00–24:00
)

repeat_count = int(np.ceil(num_steps / len(daily_dynamic_export_kw_per_phase)))

dynamic_export_profile_kw_per_phase = np.tile(
    daily_dynamic_export_kw_per_phase,
    repeat_count
)[:num_steps]

assert len(daily_dynamic_export_kw_per_phase) == 48

assert len(dynamic_export_profile_kw_per_phase) == num_steps



# Verify all profiles are the same length
profiles = [
    house_1_load,
    house_2_load,
    house_3_load,
    house_4_load,
    house_5_load,
    house_1_pv,
    house_2_pv,
    house_3_pv,
    house_4_pv,
    house_5_pv,
    dynamic_export_profile_kw_per_phase,
    hospital_load_profile,
    wind_generation_profile,
]
profile_lengths = set([len(profile) for profile in profiles])


if len(profile_lengths) > 1 :
    raise ValueError("Profiles are not the same length")
else :
    print("All Load and Generation Profiles are the same length")

print("Number of Simulation steps=", num_steps)
print("Number of Hours Simulated=", simulation_hours) 

house_loads = {
    "House 1": house_1_load,
    "House 2": house_2_load,
    "House 3": house_3_load,
    "House 4": house_4_load,
    "House 5": house_5_load,
}

house_pv = {
    "House 1": house_1_pv,
    "House 2": house_2_pv,
    "House 3": house_3_pv,
    "House 4": house_4_pv,
    "House 5": house_5_pv,
}

# ============================================================
# SCENARIO SETUP
# ============================================================

scenario_name = "coordinated_vpp_demo"

# simulation = create_coordinated_vpp_scenario(
#     house_loads=house_loads,
#     house_pv=house_pv,
#     dynamic_export_profile_kw_per_phase=(
#         dynamic_export_profile_kw_per_phase
#     ),
#     hospital_load_profile=hospital_load_profile,
#     wind_generation_profile=wind_generation_profile,
#     timestep_hours=timestep_hours,
#     num_steps=num_steps
# )

simulation = create_coordinated_vpp_scenario(
    house_loads=house_loads,
    house_pv=house_pv,
    dynamic_export_profile_kw_per_phase=(
        dynamic_export_profile_kw_per_phase
    ),
    hospital_load_profile=hospital_load_profile,
    wind_generation_profile=wind_generation_profile,
    timestep_hours=timestep_hours,
    num_steps=num_steps,
    tariff=BATTERY_LITE
)

independent_dynamic_simulation = (
    create_independent_dynamic_scenario(
        house_loads=house_loads,
        house_pv=house_pv,
        dynamic_export_profile_kw_per_phase=(
            dynamic_export_profile_kw_per_phase
        ),
        hospital_load_profile=hospital_load_profile,
        wind_generation_profile=wind_generation_profile,
        timestep_hours=timestep_hours,
        num_steps=num_steps,
        tariff=BATTERY_STARTER
    )
)

independent_fixed_simulation = (
    create_independent_fixed_scenario(
        house_loads=house_loads,
        house_pv=house_pv,
        hospital_load_profile=hospital_load_profile,
        wind_generation_profile=wind_generation_profile,
        timestep_hours=timestep_hours,
        num_steps=num_steps,
        tariff=BATTERY_STARTER,
        fixed_export_limit_kw_per_phase=5.0
    )
)




=== TIMESTAMP CHECK ===
Load start: 2013-02-04 00:00:00
Load end:   2013-02-10 23:30:00
PV start:   2013-01-07 00:00:00
PV end:     2013-01-13 23:30:00
All Load and Generation Profiles are the same length
Number of Simulation steps= 336
Number of Hours Simulated= 168.0


In [5]:
# ============================================================
# REPORT SIMULATION - 7 DAY COMPARISON
# ============================================================

from scenarios.scenario_setup import (
    create_coordinated_vpp_scenario,
    create_independent_dynamic_scenario,
    create_independent_fixed_scenario,
    COORDINATED_INPUTS,
    INDEPENDENT_INPUTS,
)

from simulation.results_export import (
    export_comparison_results
)

from economics.tariffs import BATTERY_STARTER


# ------------------------------------------------------------
# REPORT CONFIGURATION
# ------------------------------------------------------------

REPORT_DAYS = 7

report_steps = int(
    REPORT_DAYS
    * 24
    / timestep_hours
)

# Use the same tariff across all three scenarios so the
# comparison isolates the effect of coordination / PV sharing.
report_tariff = BATTERY_STARTER


# ------------------------------------------------------------
# CREATE FRESH SIMULATION OBJECTS
# ------------------------------------------------------------
#
# Important:
# Do not reuse simulations previously run by the demonstrator.
# Fresh objects reset:
#   - battery SOC
#   - financial totals
#   - current timestep
#   - controller state

simulation = create_coordinated_vpp_scenario(
    house_loads=house_loads,
    house_pv=house_pv,
    dynamic_export_profile_kw_per_phase=(
        dynamic_export_profile_kw_per_phase
    ),
    hospital_load_profile=hospital_load_profile,
    wind_generation_profile=wind_generation_profile,
    timestep_hours=timestep_hours,
    num_steps=num_steps,
    tariff=report_tariff
)

independent_dynamic_simulation = (
    create_independent_dynamic_scenario(
        house_loads=house_loads,
        house_pv=house_pv,
        dynamic_export_profile_kw_per_phase=(
            dynamic_export_profile_kw_per_phase
        ),
        hospital_load_profile=hospital_load_profile,
        wind_generation_profile=wind_generation_profile,
        timestep_hours=timestep_hours,
        num_steps=num_steps,
        tariff=report_tariff
    )
)

independent_fixed_simulation = (
    create_independent_fixed_scenario(
        house_loads=house_loads,
        house_pv=house_pv,
        hospital_load_profile=hospital_load_profile,
        wind_generation_profile=wind_generation_profile,
        timestep_hours=timestep_hours,
        num_steps=num_steps,
        tariff=report_tariff,
        fixed_export_limit_kw_per_phase=5.0
    )
)


# ------------------------------------------------------------
# FIXED REPORT INPUTS
# ------------------------------------------------------------
#
# No hardware inputs.
# No network-support events.
# No manual toggles.
#
# Coordinated case:
#   PV sharing ON
#
# Independent cases:
#   PV sharing OFF
#
# Economic export remains enabled for all three cases.

coordinated_inputs = COORDINATED_INPUTS.copy()
independent_inputs = INDEPENDENT_INPUTS.copy()


# Explicitly disable event-based behaviour for this base test.
coordinated_inputs.update({
    "network_support_request": False,
    "requested_support_kw": 0.0,
    "vpp_tariff_event": False,
})

independent_inputs.update({
    "network_support_request": False,
    "requested_support_kw": 0.0,
    "vpp_tariff_event": False,
})


# ------------------------------------------------------------
# RUN SIMULATIONS
# ------------------------------------------------------------

print(
    f"Running {REPORT_DAYS}-day report simulation "
    f"({report_steps} timesteps)..."
)

steps_per_day = int(
    24 / timestep_hours
)

for step in range(report_steps):

    simulation.step(
        coordinated_inputs
    )

    independent_dynamic_simulation.step(
        independent_inputs
    )

    independent_fixed_simulation.step(
        independent_inputs
    )

    # Simple progress output once per simulated day.
    if (
        (step + 1) % steps_per_day == 0
    ):
        completed_days = (
            (step + 1)
            // steps_per_day
        )

        print(
            f"Completed day "
            f"{completed_days}/{REPORT_DAYS}"
        )


# ------------------------------------------------------------
# EXPORT RESULTS
# ------------------------------------------------------------

filepath = export_comparison_results(
    coordinated_simulation=simulation,
    independent_dynamic_simulation=(
        independent_dynamic_simulation
    ),
    independent_fixed_simulation=(
        independent_fixed_simulation
    )
)

print(
    f"\nReport simulation complete."
)

print(
    f"Results exported to:\n{filepath}"
)

Running 7-day report simulation (336 timesteps)...
Completed day 1/7
Completed day 2/7
Completed day 3/7
Completed day 4/7
Completed day 5/7
Completed day 6/7
Completed day 7/7

Report simulation complete.
Results exported to:
results/SimulationResults_comparison_2026-10-16_01-18.xlsx


In [4]:
# --------------------------------------------------
# Continuous hardware demonstration
# --------------------------------------------------


# --------------------------------------------------
# Simulated network support event
# --------------------------------------------------

# event_day = 1
# event_hour = 22
# event_duration_hours = 2

# event_start_step = int(
#     (((event_day - 1) * 24) + event_hour)
#     / timestep_hours
# )

# event_duration_steps = int(
#     event_duration_hours / timestep_hours
# )

# event_end_step = (
#     event_start_step
#     + event_duration_steps
# )

# --------------------------------------------------
# Simulated tariff peak-shaving toggle
# --------------------------------------------------

peak_toggle_day = 2
peak_toggle_hour = 10
peak_toggle_duration_hours = 12

peak_toggle_start_step = int(
    (((peak_toggle_day - 1) * 24) + peak_toggle_hour)
    / timestep_hours
)

peak_toggle_duration_steps = int(
    peak_toggle_duration_hours / timestep_hours
)

peak_toggle_end_step = (
    peak_toggle_start_step
    + peak_toggle_duration_steps
)

hardware = HardwareInterface()


independent_inputs = {
    "network_support_request": False,
    "requested_support_kw": 0.0,
    "pv_sharing_enabled": False,
    "peak_shaving_enabled": False,
    "economic_export_enabled": True,
    "vpp_tariff_event": False,
}

try:
    print("Continuous VPP hardware demonstration started.")
    print("Use Jupyter interrupt/stop to finish the run.\n")

    while True:
        
        # Read physical ESP32 inputs first
        external_inputs = hardware.read_inputs()
        
        # ----------------------------------------------
        # ECONOMIC PEAK EXPORT
        # ----------------------------------------------
        #
        # Temporary demonstration input.
        # Enables deliberate battery export during the
        # peak hours configured in VPPController.
        
        external_inputs["economic_export_enabled"] = True
        external_inputs["peak_shaving_enabled"] = False

        # external_inputs["vpp_tariff_event"] = True


        
        # ----------------------------------------------
        # DAILY REPEATING PV SHARING
        # ----------------------------------------------
    
        current_hour_of_day = (
            simulation.current_step * timestep_hours
        ) % 24
    
        pv_sharing_enabled = (
            8.0 <= current_hour_of_day < 16.0
        )
    
        external_inputs["pv_sharing_enabled"] = (
            pv_sharing_enabled
        )
      

        result = simulation.step(external_inputs)


        independent_dynamic_result = (
            independent_dynamic_simulation.step(
                independent_inputs
            )
        )
        
        independent_fixed_result = (
            independent_fixed_simulation.step(
                independent_inputs
            )
        )

      
        # Convert latest simulation results
        # into LED states
        hardware_states = get_hardware_states(simulation)

        

        hardware.update_outputs(
            result,
            hardware_states
        )
        

        # Print useful live information
        print(
            f"\nStep {result['Simulation_step']} | "
            f"Day {result['Day_number']} | "
            f"Hour {result['Hour_of_day']:.1f} | "
            f"Mode: {result['VPP_mode']} | "
            f"Network support: "
            f"{'ON' if external_inputs.get('network_support_request', False) else 'OFF'} | "
            f"Peak strategy: {'ON' if  external_inputs.get('peak_shaving_enabled', False) else 'OFF'} | "
            f"Economic export: "
            f"{'ON' if external_inputs.get('economic_export_enabled', False) else 'OFF'} | "
            f"PV sharing: {'ON' if result['PV_sharing_enabled'] else 'OFF'} | "
        )

        for house in simulation.houses:

            latest = simulation.house_results_log[house.name][-1]
            state = hardware_states[house.name]["state"]

            print(
                f"{house.name}: "
                f"{state:12s} | "
                f"Load = {latest['load_kw']:.2f} kW | "
                f"PV = {latest['generation_kw']:.2f} kW | "
                f"Grid = {latest['net_power_kw']:.2f} kW | "
                f"SOC = {latest['battery_soc_kwh']:.2f} kWh "
                f"({latest['battery_soc_percent']:.1f}%) |"
                f"Charge = {latest['battery_charge_kw']:.2f} kW | "
                f"Discharge = {latest['battery_discharge_kw']:.2f} kW"
            )
        print(
            "Shared PV available: "
            f"{result['Shared_PV_available_kw']:.2f} kW | "
            f"Shared PV targets: {result['house_shared_pv_targets']}"
        )

        # Extra output during network-support event
        if result["VPP_mode"] == "network_support":

            print(
                "NETWORK SUPPORT | "
                f"Requested = {result['Requested_support_kw']:.2f} kW | "
                f"Available = {result['Available_VPP_discharge_kw']:.2f} kW | "
                f"Actual = {result['Battery_discharge_kw']:.2f} kW"
            )


            
        time.sleep(1)


except KeyboardInterrupt:

    print("\nSimulation interrupted by user.")


finally:

    hardware.cleanup()

    print("Hardware interface cleaned up.")

# --------------------------------------------------
# Export everything accumulated during the run
# --------------------------------------------------

filepath = export_comparison_results(
    coordinated_simulation=simulation,
    independent_dynamic_simulation=(
        independent_dynamic_simulation
    ),
    independent_fixed_simulation=(
        independent_fixed_simulation
    )
)

print(
    f"Results exported to {filepath}"
)

Using ESP32 serial port: /dev/ttyACM0
Continuous VPP hardware demonstration started.
Use Jupyter interrupt/stop to finish the run.


Step 0 | Day 1 | Hour 0.0 | Mode: self_consumption | Network support: OFF | Peak strategy: OFF | Economic export: ON | PV sharing: OFF | 
House 1: discharging  | Load = 0.71 kW | PV = 0.00 kW | Grid = 0.00 kW | SOC = 12.77 kWh (48.6%) |Charge = -0.00 kW | Discharge = 0.71 kW
House 2: discharging  | Load = 0.15 kW | PV = 0.00 kW | Grid = 0.00 kW | SOC = 8.68 kWh (49.5%) |Charge = -0.00 kW | Discharge = 0.15 kW
House 3: no_battery   | Load = 0.14 kW | PV = 0.00 kW | Grid = 0.14 kW | SOC = 0.00 kWh (0.0%) |Charge = -0.00 kW | Discharge = 0.00 kW
House 4: discharging  | Load = 0.21 kW | PV = 0.00 kW | Grid = 0.00 kW | SOC = 13.03 kWh (49.6%) |Charge = -0.00 kW | Discharge = 0.21 kW
House 5: discharging  | Load = 0.35 kW | PV = 0.00 kW | Grid = 0.00 kW | SOC = 21.71 kWh (49.6%) |Charge = -0.00 kW | Discharge = 0.35 kW
Shared PV available: 0.00 kW | Shared PV t